# Scraping Wikipedia Pages on Jewish Names

Author: Kyla Chasalow, with substantial help from ChatGPT-4 (in November 2024) and ChatGPT-5 (in November 2025)

General Imports

In [8]:
import numpy as np
import pandas as pd
import os
import sys
import copy
import pprint
import re
import csv
import requests
from bs4 import BeautifulSoup

# November 2024
Old approach that worked in November 2024 but not now after Wikipedia changed scraping settings

In [14]:
def scrape_wikipedia_surnames(url):
    # Send a GET request to the URL
    response = requests.get(url)
    if response.status_code != 200:
        print(f"Failed to retrieve the page. Status code: {response.status_code}")
        return None
    
    # Parse the HTML content of the page
    soup = BeautifulSoup(response.text, 'html.parser')
    
    # Find the section containing the list of surnames
    surname_list = []
    div_content = soup.find('div', {'id': 'mw-pages'})
    if div_content:
        # Extract the list items
        for link in div_content.find_all('a'): #locate anchor
            surname = link.text #retrieve visible text (e.g., title = "Gottfried")
            surname_list.append(surname)
    
    return surname_list

def clean_surnames(surnames):

    # Remove "previous page" and "next page"
    filtered_surnames = [surname for surname in surnames if "page" not in surname.lower()]
    filtered_surnames.remove("This list may not reflect recent changes")
    
    # Remove extra text in ( ) and remove any extra whitespace
    cleaned_surnames = [re.sub(r"\s*\(.*?\)", "", surname).strip() for surname in filtered_surnames]

    # Remove the word family
    cleaned_surnames = [re.sub(r"family", "", surname).strip() for surname in cleaned_surnames]

    # Make them all caps
    cleaned_surnames = [name.upper() for name in cleaned_surnames] 

    return(cleaned_surnames)

In [16]:
base_url = "https://en.wikipedia.org/wiki/Category:Surnames_of_Jewish_origin?from="

# Generate URLs for each letter from A to Z
# Note that these are "from = " urls so actually Y already gets me Z's and then Z gets me Z's so  need to deduplicate in cleaning
url_list = [f"{base_url}{chr(letter)}" for letter in range(ord('A'), ord('Z') + 1)]

final_surname_list = []

for url in url_list:

    # Scrape the surnames
    surnames = scrape_wikipedia_surnames(url)

    # Clean them
    surnames = clean_surnames(surnames)

    final_surname_list += surnames


# Remove duplicates that arise from from= aspect of links
final_surname_list = list(set(final_surname_list))
final_surname_list.sort()

In [50]:
# Save to CSV file (overwrites if the file already exists)
csv_filename = "WikipediaSurnames20251125.csv"
with open(csv_filename, mode='w', newline='', encoding='utf-8') as csv_file:
    writer = csv.writer(csv_file)
    
    # Write the header
    writer.writerow(["surname"])
    
    # Write the surnames
    for surname in final_surname_list:
        writer.writerow([surname])

print(f"Surnames successfully saved to {csv_filename} (previous file erased if it existed).")

Surnames successfully saved to surnames.csv (previous file erased if it existed).


# November 2025
Written by ChatGPT to deal with new needs. Reviewed to make sure outputs looked correct.

In [39]:
import time
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from bs4 import BeautifulSoup
import re

SESSION = requests.Session()
retries = Retry(
    total=5,
    backoff_factor=1.5,
    status_forcelist=[403, 429, 500, 502, 503, 504],
    allowed_methods=["GET"]
)
SESSION.mount("https://", HTTPAdapter(max_retries=retries))

HEADERS = {
    "User-Agent": "SurnameScraper/1.0 (+youremail@example.com) Requests"
}

def scrape_wikipedia_surnames(url):
    resp = SESSION.get(url, headers=HEADERS, timeout=20)
    if resp.status_code != 200:
        raise RuntimeError(f"Failed to retrieve the page. Status code: {resp.status_code}")

    soup = BeautifulSoup(resp.text, "html.parser")
    surname_list = []
    div_content = soup.find("div", {"id": "mw-pages"})
    if div_content:
        for link in div_content.find_all("a"):
            surname_list.append(link.text)
    return surname_list

def clean_surnames(surnames):
    # Guard against missing message
    blocked = ["this list may not reflect recent changes", "jewish name", "kinnui"]
    filtered = [s for s in surnames if "page" not in s.lower() and s.lower() not in blocked]
    cleaned = [re.sub(r"\s*\(.*?\)", "", s).strip() for s in filtered]
    cleaned = [re.sub(r"\bfamily\b", "", s, flags=re.I).strip() for s in cleaned]
    return [s.upper() for s in cleaned if s]

In [40]:
base_url = "https://en.wikipedia.org/wiki/Category:Surnames_of_Jewish_origin?from="
url_list = [f"{base_url}{chr(letter)}" for letter in range(ord('A'), ord('Z') + 1)]

final_surname_list = []
for url in url_list:
    print(url)
    names = scrape_wikipedia_surnames(url)
    final_surname_list.extend(clean_surnames(names))
    time.sleep(1.0)  # be polite

final_surname_list = sorted(set(final_surname_list)) #de-duplicate

https://en.wikipedia.org/wiki/Category:Surnames_of_Jewish_origin?from=A
https://en.wikipedia.org/wiki/Category:Surnames_of_Jewish_origin?from=B
https://en.wikipedia.org/wiki/Category:Surnames_of_Jewish_origin?from=C
https://en.wikipedia.org/wiki/Category:Surnames_of_Jewish_origin?from=D
https://en.wikipedia.org/wiki/Category:Surnames_of_Jewish_origin?from=E
https://en.wikipedia.org/wiki/Category:Surnames_of_Jewish_origin?from=F
https://en.wikipedia.org/wiki/Category:Surnames_of_Jewish_origin?from=G
https://en.wikipedia.org/wiki/Category:Surnames_of_Jewish_origin?from=H
https://en.wikipedia.org/wiki/Category:Surnames_of_Jewish_origin?from=I
https://en.wikipedia.org/wiki/Category:Surnames_of_Jewish_origin?from=J
https://en.wikipedia.org/wiki/Category:Surnames_of_Jewish_origin?from=K
https://en.wikipedia.org/wiki/Category:Surnames_of_Jewish_origin?from=L
https://en.wikipedia.org/wiki/Category:Surnames_of_Jewish_origin?from=M
https://en.wikipedia.org/wiki/Category:Surnames_of_Jewish_origin

In [41]:
# Save to CSV file (overwrites if the file already exists)
csv_filename = "WikipediaSurnames20251105.csv"
with open(csv_filename, mode='w', newline='', encoding='utf-8') as csv_file:
    writer = csv.writer(csv_file)
    
    # Write the header
    writer.writerow(["surname"])
    
    # Write the surnames
    for surname in final_surname_list:
        writer.writerow([surname])

print(f"Surnames successfully saved to {csv_filename} (previous file erased if it existed).")

Surnames successfully saved to WikipediaSurnames20251105.csv (previous file erased if it existed).


# First Names

In [49]:
url_list = ["https://en.wikipedia.org/wiki/Category:Jewish_given_names",
            "https://en.wikipedia.org/wiki/Category:Hebrew-language_given_names",
            "https://en.wikipedia.org/wiki/Category:Yiddish-language_given_names",
            "https://en.wikipedia.org/wiki/Category:Hebrew_unisex_given_names",
            "https://en.wikipedia.org/wiki/Category:Jewish_masculine_given_names", #masculine
            "https://en.wikipedia.org/wiki/Category:Hebrew_masculine_given_names",
            "https://en.wikipedia.org/wiki/Category:Yiddish_masculine_given_names",
            "https://en.wikipedia.org/wiki/Category:Jewish_feminine_given_names", #feminine
            "https://en.wikipedia.org/wiki/Category:Hebrew_feminine_given_names",
            "https://en.wikipedia.org/wiki/Category:Yiddish_feminine_given_names",
  ]

In [50]:
first_name_list = []
for url in url_list:
    print(url)
    names = scrape_wikipedia_surnames(url)
    first_name_list.extend(clean_surnames(names))
    time.sleep(1.0)  # be polite

https://en.wikipedia.org/wiki/Category:Jewish_given_names
https://en.wikipedia.org/wiki/Category:Hebrew-language_given_names
https://en.wikipedia.org/wiki/Category:Yiddish-language_given_names
https://en.wikipedia.org/wiki/Category:Hebrew_unisex_given_names
https://en.wikipedia.org/wiki/Category:Jewish_masculine_given_names
https://en.wikipedia.org/wiki/Category:Hebrew_masculine_given_names
https://en.wikipedia.org/wiki/Category:Yiddish_masculine_given_names
https://en.wikipedia.org/wiki/Category:Jewish_feminine_given_names
https://en.wikipedia.org/wiki/Category:Hebrew_feminine_given_names
https://en.wikipedia.org/wiki/Category:Yiddish_feminine_given_names


In [51]:
first_name_list = sorted(set(first_name_list))
print(first_name_list[0:10])
len(first_name_list)

['AARON', 'ABBA', 'ABEL', 'ABIGAIL', 'ABNER', 'ABO', 'ABRAHAM', 'ABRAM', 'ABU AL-FARAJ', 'ADAM']


388

In [52]:
# Save to CSV file (overwrites if the file already exists)
csv_filename = "../../Data/Wikipedia/WikipediaFirstNames20251105.csv"
with open(csv_filename, mode='w', newline='', encoding='utf-8') as csv_file:
    writer = csv.writer(csv_file)
    
    # Write the header
    writer.writerow(["first"])
    
    # Write the surnames
    for name in first_name_list:
        writer.writerow([name])

print(f"Surnames successfully saved to {csv_filename} (previous file erased if it existed).")

Surnames successfully saved to ../../Data/Wikipedia/WikipediaFirstNames20251105.csv (previous file erased if it existed).
